In [2]:
import pandas as pd

In [3]:
df_raw = pd.read_csv(r'C:\Users\moham\Desktop\Data Science\BeCode\becode_projects\immo-eliza\data\raw\listings_with_distance.csv')

In [4]:
df = df_raw.copy()

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26349 entries, 0 to 26348
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                26157 non-null  str    
 1   property_type           26349 non-null  str    
 2   subtype                 26349 non-null  str    
 3   price_eur               25708 non-null  float64
 4   type_of_sale            26349 non-null  str    
 5   num_rooms               25614 non-null  float64
 6   living_area_m2          24360 non-null  float64
 7   fully_equipped_kitchen  8508 non-null   float64
 8   furnished               15450 non-null  float64
 9   terrace                 24624 non-null  float64
 10  terrace_area_m2         10410 non-null  float64
 11  garden                  24023 non-null  float64
 12  garden_area_m2          5917 non-null   float64
 13  land_surface_m2         13494 non-null  float64
 14  num_facades             19940 non-null  float64
 

In [6]:
#removing duplicates
df = df.drop_duplicates()
df.shape

(25698, 20)

In [7]:
#removing rows with no price because irrelevant
df = df.dropna(subset=['price_eur'])
df.shape

(25255, 20)

In [8]:
df['locality'].unique()[:10]
df['property_type'].unique()[:10]
df['subtype'].unique()

<StringArray>
[         'Villa',      'Residence',      'Apartment',   'Ground Floor',
         'Chalet', 'Mixed Building',      'Penthouse',         'Studio',
        'Triplex',         'Duplex',        'Mansion',        'Cottage',
   'Master House',           'Loft',       'Bungalow']
Length: 15, dtype: str

In [10]:
# fill columns with missing values with 0 (Assumption: Missing = Absent)
columns_to_fill = ['swimming_pool', 'furnished', 'terrace', 'fully_equipped_kitchen', 'garden']
df[columns_to_fill] = df[columns_to_fill].fillna(0)
df.info()

<class 'pandas.DataFrame'>
Index: 25255 entries, 0 to 26348
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                25255 non-null  str    
 1   property_type           25255 non-null  str    
 2   subtype                 25255 non-null  str    
 3   price_eur               25255 non-null  float64
 4   type_of_sale            25255 non-null  str    
 5   num_rooms               24727 non-null  float64
 6   living_area_m2          23556 non-null  float64
 7   fully_equipped_kitchen  25255 non-null  float64
 8   furnished               25255 non-null  float64
 9   terrace                 25255 non-null  float64
 10  terrace_area_m2         10019 non-null  float64
 11  garden                  25255 non-null  float64
 12  garden_area_m2          5722 non-null   float64
 13  land_surface_m2         13060 non-null  float64
 14  num_facades             19292 non-null  float64
 15  s

In [11]:
'''
Critical Thinking Challenge:
You have ~25,000 rows. If you drop the 1,693 rows where living_area_m2 is missing, you still have ~23,000 rows.
Is it better to have 25,000 rows with 6% "guessed" data, or 23,000 rows of "pure" data?
Think about the "Signal-to-Noise" ratio. Which choice makes your future model more trustworthy?
'''
df = df.dropna(subset=['living_area_m2'])
df.shape
df.info()

<class 'pandas.DataFrame'>
Index: 23556 entries, 0 to 26348
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                23556 non-null  str    
 1   property_type           23556 non-null  str    
 2   subtype                 23556 non-null  str    
 3   price_eur               23556 non-null  float64
 4   type_of_sale            23556 non-null  str    
 5   num_rooms               23187 non-null  float64
 6   living_area_m2          23556 non-null  float64
 7   fully_equipped_kitchen  23556 non-null  float64
 8   furnished               23556 non-null  float64
 9   terrace                 23556 non-null  float64
 10  terrace_area_m2         9873 non-null   float64
 11  garden                  23556 non-null  float64
 12  garden_area_m2          5607 non-null   float64
 13  land_surface_m2         12284 non-null  float64
 14  num_facades             18193 non-null  float64
 15  s

In [12]:
# A house with "NaN" bedrooms is an incomplete listing. Since you have over 23,000 clean rows, you do not need these 500 "broken" rows.
# sacrifice quantity for quality
print(df['num_rooms'].value_counts().head(10))
df = df.dropna(subset=['num_rooms'])
print(f"{df.shape=}")

num_rooms
3.0    7842
2.0    6722
4.0    3709
1.0    2370
5.0    1410
6.0     524
7.0     222
0.0     137
8.0      91
9.0      47
Name: count, dtype: int64
df.shape=(23187, 20)


In [13]:
'''
We will set a lower bound. 
A house in Belgium for less than €10,000 or smaller than 10m² 
is likely an error or a parking spot, 
not a residential property for our analysis.
'''
df = df[df['price_eur'] > 10000]
df = df[df['living_area_m2'] > 10]
print(f"{df.shape=}")
df.info()

df.shape=(23169, 20)
<class 'pandas.DataFrame'>
Index: 23169 entries, 0 to 26348
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                23169 non-null  str    
 1   property_type           23169 non-null  str    
 2   subtype                 23169 non-null  str    
 3   price_eur               23169 non-null  float64
 4   type_of_sale            23169 non-null  str    
 5   num_rooms               23169 non-null  float64
 6   living_area_m2          23169 non-null  float64
 7   fully_equipped_kitchen  23169 non-null  float64
 8   furnished               23169 non-null  float64
 9   terrace                 23169 non-null  float64
 10  terrace_area_m2         9800 non-null   float64
 11  garden                  23169 non-null  float64
 12  garden_area_m2          5586 non-null   float64
 13  land_surface_m2         12171 non-null  float64
 14  num_facades             17971 non

In [15]:
''''terrace_area_m2 and garden_area_m2 missing values.
If a row has no terrace or garden, 
the terrace_area_m2 and garden_area_m2 are naturally NaN or 0.
'''
df.loc[df['terrace'] == 0, 'terrace_area_m2'] = 0
df.loc[df['garden'] == 0, 'garden_area_m2'] = 0

'''You have ~11,000 missing values in land_surface_m2
Critical Thinking: Does an apartment have "land surface"? 
No. Apartments exist in buildings; 
the "land" belongs to the co-ownership.
If property_type == "Apartment", 
set land_surface_m2 to 0. 
If property_type == "House",
LEAVE AS NAN FOR NOW, WILL DO .FILLNA FOR MODEL
'''
df.loc[(df['property_type'] == 'Apartment') & (df['land_surface_m2'].isnull()), 'land_surface_m2'] = 0

df.info()

<class 'pandas.DataFrame'>
Index: 23169 entries, 0 to 26348
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                23169 non-null  str    
 1   property_type           23169 non-null  str    
 2   subtype                 23169 non-null  str    
 3   price_eur               23169 non-null  float64
 4   type_of_sale            23169 non-null  str    
 5   num_rooms               23169 non-null  float64
 6   living_area_m2          23169 non-null  float64
 7   fully_equipped_kitchen  23169 non-null  float64
 8   furnished               23169 non-null  float64
 9   terrace                 23169 non-null  float64
 10  terrace_area_m2         15626 non-null  float64
 11  garden                  23169 non-null  float64
 12  garden_area_m2          15292 non-null  float64
 13  land_surface_m2         21441 non-null  float64
 14  num_facades             17971 non-null  float64
 15  s

In [16]:
#num_facades missing values logic
'''
most houses have 2 facades (terraced/row houses). 
By filling with the median, you are saying: 
"In the absence of info, I will assume this is the most common type of house."
If you use the Median (2.0 in your dataset), 
you are making a "Probabilistic Bet."
Most Machine Learning models (like Linear Regression) cannot handle NaNs.
The model will delete the whole row!
The Verdict: We use the Median because the "damage" done by a slight guess 
is much smaller than the "damage" of losing 6,000 rows 
of perfectly good price and area data.
LEAVE AS NAN FOR NOW, WILL DO .MEDIAN FOR MODEL:
df['num_facades'] = df['num_facades'].fillna(df['num_facades'].median())
'''
print(f"{df['num_facades'].value_counts()=}")






df['num_facades'].value_counts()=num_facades
2.0    8301
4.0    4925
3.0    4706
1.0      39
Name: count, dtype: int64


In [22]:
#zip_code column creation
#zip_code extraction from locality column
df['zip_code'] = df['locality'].str.extract(r'(\d{4})').astype(float).astype('int64')

#zip_code column reordering to index 1
# Get a list of all current columns
cols = list(df.columns)
# Move the last element (zip_code) to the second position
cols.insert(1, cols.pop(cols.index('zip_code')))
df = df[cols]

print(df[['locality', 'zip_code']].head())

               locality  zip_code
0           7130 Binche      7130
1    3190 Boortmeerbeek      3190
2        2350 Vosselaar      2350
3   4460 Grâce-Hollogne      4460
4  1420 Braine-l'Alleud      1420


In [23]:
#convert columns to int64 where decimals is impossible (rooms, facades, binary 0/1)
#cleaner and uses less memory
cols_to_int = ['num_rooms', 'fully_equipped_kitchen', 'furnished', 'terrace', 'garden', 'swimming_pool', 'num_facades']
df[cols_to_int] = df[cols_to_int].astype('Int64')
df.info()

<class 'pandas.DataFrame'>
Index: 23169 entries, 0 to 26348
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                23169 non-null  str    
 1   zip_code                23169 non-null  int64  
 2   property_type           23169 non-null  str    
 3   subtype                 23169 non-null  str    
 4   price_eur               23169 non-null  float64
 5   type_of_sale            23169 non-null  str    
 6   num_rooms               23169 non-null  Int64  
 7   living_area_m2          23169 non-null  float64
 8   fully_equipped_kitchen  23169 non-null  Int64  
 9   furnished               23169 non-null  Int64  
 10  terrace                 23169 non-null  Int64  
 11  terrace_area_m2         15626 non-null  float64
 12  garden                  23169 non-null  Int64  
 13  garden_area_m2          15292 non-null  float64
 14  land_surface_m2         21441 non-null  float64
 15  n

In [24]:
#funct takes a zip code and returns the name of the Region (Flanders, Wallonia, or Brussels)
def get_region(zip_code):
    if 1000 <= zip_code <= 1299:
        return 'Brussels'
    elif ( 1300 <= zip_code <= 1499) or ( 4000 <= zip_code <= 7999 ):
        return 'Wallonia'
    else:
        return 'Flanders'
    



In [30]:
#creating the region column 
df['region'] = df['zip_code'].apply(get_region)

#moving column region to index 1
cols = list(df.columns)
cols.insert(1, cols.pop(cols.index('region')))
df = df[cols]

df['region'].value_counts()
df.columns
df.info()

<class 'pandas.DataFrame'>
Index: 23169 entries, 0 to 26348
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   locality                23169 non-null  str    
 1   region                  23169 non-null  str    
 2   zip_code                23169 non-null  int64  
 3   property_type           23169 non-null  str    
 4   subtype                 23169 non-null  str    
 5   price_eur               23169 non-null  float64
 6   type_of_sale            23169 non-null  str    
 7   num_rooms               23169 non-null  Int64  
 8   living_area_m2          23169 non-null  float64
 9   fully_equipped_kitchen  23169 non-null  Int64  
 10  furnished               23169 non-null  Int64  
 11  terrace                 23169 non-null  Int64  
 12  terrace_area_m2         15626 non-null  float64
 13  garden                  23169 non-null  Int64  
 14  garden_area_m2          15292 non-null  float64
 15  l

In [33]:
df.to_csv(r'C:\Users\moham\Desktop\Data Science\BeCode\becode_projects\immo-eliza\data\processed\listings_with_distance_clean.csv', index=False)